In [ ]:
import socket
import threading
import time

MAX_CLIENTS = 4
client_count = 0
lock = threading.Lock()

def handle_client(conn, addr):
    global client_count
    print(f"[+] Client {addr} connected.")

    try:
        # last_send_str = ""
        while True:
            # 接受客户端消息
            data = conn.recv(1024).decode()
            print(f"[{addr}] Received: {data}")

            # print(type(data))
            response = f"Hello from server! You said: {data}"

            if data == "Want SNs":
                response = "SN1#DLCHKV000080001BJT#,SN2#DLCHKV000090001BJT#,SN3#DLCHKV000070001BJT#,SN4#DLCHKV000060001BJT#,"

            # 回复客户端消息
            # if last_send_str == response:
            #     continue

            print(f"[{addr}] Sending: {response}")
            conn.sendall(response.encode())
            # last_send_str = response

        # Simulate long processing time
        # time.sleep(180 * 4) # 模拟处理时间....

    except Exception as e:
        print(f"[!] Error with client {addr}: {e}")
    finally:
        # 关闭连接
        conn.close()
        with lock:
            global client_count
            client_count -= 1
        print(f"[-] Client {addr} disconnected. Current Active Clients: {client_count}")


def main():
    global client_count

    """
    socket.AF_INET: address family, IPv4
    socket.SOCK_STREAM: socket type, TCP
    """
    # 创建套接字对象
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    """
    socket.SOL_SOCKET: 这是一个选项级别，表示选项在套接字级别上进行设置。SOL_SOCKET 是一个常量，用于指定选项的层次，即套接字级别的选项。
    socket.SO_REUSEADDR: 允许重用本地地址和端口
    """
    # 设置套接字选项
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    # 绑定地址和端口
    server.bind(('127.0.0.1',12345))
    # 监听连接
    server.listen(MAX_CLIENTS)
    print(f"🚀 Server started on 127.0.0.1:12345, Max Clients: {MAX_CLIENTS}...")
    try:
        while True:
            # 等待客户端连接
            conn, addr = server.accept()
            # 检查当前客户端数量
            with lock:
                if client_count >= MAX_CLIENTS:
                    print(f"[!] Max clients{MAX_CLIENTS} reached. Rejecting connection from {addr}.")
                    conn.close()
                    continue
                client_count += 1
            # Creating a new thread for each client connection
            client_thread = threading.Thread(target=handle_client, args=(conn, addr))
            client_thread.daemon = True # Ensures thread exits when main program exits
            client_thread.start()
            print(f"[+] Current Active Clients: {client_count}")
    except KeyboardInterrupt:
        print("\n[!] Server shutting down...")
    finally:
        server.close()

if __name__ == '__main__':
    main()

nevertheless